## Concept 6 — Tool & Function Calling

### What is it?
Tool calling lets the LLM decide **which function to call and with what arguments** — instead of
answering directly, it outputs a structured tool invocation that your code executes.

### Pipeline
```
Human → LLM decides: which tool + what args
       → your code executes the tool
       → ToolMessage sent back to LLM
       → LLM forms final answer using tool result
```

### Limitation (what Concept 7 fixes)
The LLM jumps to action without showing its reasoning. Hard to debug and can pick wrong tools.
→ Concept 7 adds ReAct: the LLM writes its thoughts before each action.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

llm = get_llm(temperature=0.0)
print('LLM ready')

### Step 1 — Define Tools
The `@tool` decorator turns a Python function into a LangChain tool.
The docstring becomes the tool description the LLM reads to decide when to use it.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # Mocked — replace with real API call
    weather_data = {
        'Hyderabad': '34C, Sunny',
        'Mumbai':    '29C, Humid',
        'Delhi':     '27C, Cloudy',
        'Bangalore': '22C, Pleasant',
    }
    return weather_data.get(city, f'No data for {city}')

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        result = eval(expression, {'__builtins__': {}}, {})
        return str(result)
    except Exception as e:
        return f'Error: {e}'

@tool
def word_count(text: str) -> str:
    """Count the number of words in a text string."""
    return str(len(text.split()))

tools = [get_weather, calculate, word_count]
print(f'Tools defined: {[t.name for t in tools]}')
print(f'\nget_weather description: {get_weather.description}')

### Step 2 — Bind Tools to LLM
`bind_tools` tells the LLM which tools are available. The LLM decides whether to call one.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

# Ask a question that requires a tool
response = llm_with_tools.invoke('What is the weather in Hyderabad?')

print(f'Response type: {type(response).__name__}')
print(f'Content: "{response.content}"')
print(f'Tool calls: {response.tool_calls}')

### Step 3 — Execute the Tool and Return Result
Your code runs the tool. The result is sent back to LLM as a `ToolMessage`.

In [ ]:
# Map tool name to function
tool_map = {t.name: t for t in tools}

def execute_tool_calls(response):
    """Execute all tool calls in a response and return ToolMessages."""
    tool_messages = []
    for call in response.tool_calls:
        tool_fn = tool_map[call['name']]
        result  = tool_fn.invoke(call['args'])
        print(f'  Executed: {call["name"]}({call["args"]}) → {result}')
        tool_messages.append(
            ToolMessage(content=result, tool_call_id=call['id'])
        )
    return tool_messages

tool_msgs = execute_tool_calls(response)
print(f'\nTool results: {[m.content for m in tool_msgs]}')

### Step 4 — Full Tool Loop
Send tool results back to LLM. It forms the final answer using the real data.

In [ ]:
def run_with_tools(user_input: str) -> str:
    """Full tool call loop: query → tool decision → execute → final answer."""
    messages = [HumanMessage(content=user_input)]

    # Step 1: LLM decides tool
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    # Step 2: If tool calls exist, execute them
    if response.tool_calls:
        tool_results = execute_tool_calls(response)
        messages.extend(tool_results)

        # Step 3: LLM forms final answer with tool results
        final = llm_with_tools.invoke(messages)
        return final.content

    return response.content  # No tool needed — answered directly

# Test various queries
queries = [
    'What is the weather in Mumbai?',
    'What is 128 * 256?',
    'How many words in: The quick brown fox jumps over the lazy dog',
    'What is the capital of France?',  # No tool needed
]

print('Tool Calling Results:')
for q in queries:
    print(f'\nQ: {q}')
    print(f'A: {run_with_tools(q)}')
    print('-' * 40)

### Bonus — Parallel Tool Calls
The LLM can call multiple tools in a single response.

In [ ]:
multi_q = 'What is the weather in Delhi and Bangalore? Also calculate 99 * 99.'
print(f'Query: {multi_q}\n')
print(f'Answer: {run_with_tools(multi_q)}')